In [0]:

dbutils.widgets.text("env", "")
env = dbutils.widgets.get("env")

dbutils.widgets.text("source_table","")
source_table = dbutils.widgets.get("source_table")


#read config
from pyspark.sql.functions import col

config = (
    spark.read.table("workspace.default.config_environment")
         .filter(col("env") == env)
         .collect()[0]
)

catalog = config["catalog"]
schema = config["schema"]
secret_scope = config["secret_scope"]

# #currently free edition doesnt allow to create scopes, but this is how we fetch it after admin creates a scope like this , with 'key'(sql-password) and 'value'(password123)
# password = dbutils.secrets.get(
#     scope=secret_scope,
#     key="sql-password"
# )


print(f"Environment : {env}")
print(f"Catalog     : {catalog}")
print(f"Schema      : {schema}")
print(f"Table       : {source_table}")

#Create a reusable function to load data into Bronze
from pyspark.sql.functions import current_timestamp, lit

def load_to_bronze(source_table_name):
    
    df = spark.read.table(f"{catalog}.default.{source_table_name}")

    bronze_df = (
        df
        .withColumn("ingestion_ts", current_timestamp())
        .withColumn("source_table", lit(source_table_name))
        .withColumn("load_timestamp",current_timestamp())
    )

    (
        bronze_df
        .write
        .format("delta")
        .mode("overwrite")
        # .saveAsTable(f"workspace.default.bronze_{source_table_name}")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.bronze_{source_table_name}")
    )

    print(f"Loaded bronze_{source_table_name}")

#Drop tables
# spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_customers")
# spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_products")
# spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_orders")

load_to_bronze(source_table)

#display(spark.read.table("workspace.default.bronze_orders"))

